# SpikeDistill-RUL final experiments

Run in a Colab GPU runtime. All four subsets and all three seeds use the original baseline data preparation and the validation-selected attention BiLSTM checkpoints. Student configurations are fixed before official test scoring. Drive saves a checkpoint after each epoch.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import torch
from google.colab import drive

drive.mount("/content/drive")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU under Runtime > Change runtime type.")
project = Path("/content/drive/MyDrive/ML projec")
raw = project / "CMaps"
runs = project / "SpikeDistill-RUL" / "colab_runs"
baselines = runs / "baselines_v1"
output = runs / "spikedistill_v1"
data_root = Path("/content/spikedistill_baselines_data")
repo = Path("/content/spikedistill_baseline_source")
subsets = ["FD001", "FD002", "FD003", "FD004"]
seeds = ["42", "43", "44"]
epochs = "30"
revision = (baselines / "git_revision.txt").read_text().strip()
assert raw.is_dir(), raw
assert (baselines / "train_summary.json").is_file()
output.mkdir(parents=True, exist_ok=True)

def run(*command):
    with subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)

if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", "https://github.com/abhav-arora/ML3103-project.git", str(repo)], check=True)
run("git", "checkout", "--detach", revision)
(output / "environment.json").write_text(json.dumps({
    "python": sys.version, "torch": str(torch.__version__), "gpu": torch.cuda.get_device_name(0),
    "cuda": torch.version.cuda, "baseline_revision": revision
}, indent=2))
run(sys.executable, "prepare_data.py", "--data-dir", str(raw), "--output-dir", str(data_root), "--subsets", *subsets)


In [ ]:
# Materialize the student source bundled with this notebook.
student_source = '"""Train matched spiking students and compare their frozen checkpoints."""\nimport argparse\nimport json\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence\n\nfrom baseline_models import HardSpike, Sequence\nfrom metrics import regression_metrics\nfrom run_baselines import aggregate, batches, predict_cycles, read_split, save_result, write_json\nfrom train_pilot import digest, load_latest, restore_rng, rng_state, save_checkpoint\n\nVARIANTS = {\n    "learnable_snn": (1.0, 0.0, 0.0, 0.0, 0.0),\n    "spike_output_kd": (0.5, 0.5, 0.0, 0.0, 0.0),\n    "spike_temporal_kd": (0.5, 0.5, 0.1, 0.1, 0.0),\n    "spikedistill": (0.5, 0.5, 0.1, 0.1, 0.01),\n}\n\n\nclass Teacher(Sequence):\n    """Expose the same BiLSTM states and attention used by the baseline."""\n    def __init__(self, features):\n        super().__init__(features, "attention_bilstm")\n\n    def signals(self, x, mask):\n        order = torch.argsort(~mask, dim=1, stable=True)\n        observed = (x * mask.unsqueeze(-1)).gather(1, order.unsqueeze(-1).expand_as(x))\n        lengths = mask.sum(1)\n        packed = pack_padded_sequence(observed, lengths.cpu(), batch_first=True, enforce_sorted=False)\n        output, _ = self.cell(packed)\n        states, _ = pad_packed_sequence(output, batch_first=True, total_length=x.size(1))\n        valid = torch.arange(x.size(1), device=x.device)[None, :] < lengths[:, None]\n        weights = self.attention(states).squeeze(-1).masked_fill(~valid, -torch.inf).softmax(1)\n        prediction = self.readout((states * weights.unsqueeze(-1)).sum(1)).squeeze(-1)\n        inverse = order.argsort(1)\n        return prediction, states.gather(1, inverse.unsqueeze(-1).expand_as(states)), weights.gather(1, inverse)\n\n\nclass Student(nn.Module):\n    def __init__(self, features, width=48):\n        super().__init__()\n        self.input = nn.Linear(features, width)\n        self.recurrent = nn.Linear(width, width, bias=False)\n        self.leak_logits = nn.Parameter(torch.full((width,), float(np.log(9))))\n        self.threshold_logits = nn.Parameter(torch.full((width,), float(np.log(np.expm1(1 - 1e-4)))))\n        self.attention = nn.Linear(width, 1, bias=False)\n        self.readout = nn.Linear(width, 1)\n\n    def signals(self, x, mask):\n        currents = self.input(x)\n        voltage = currents.new_zeros(currents.size(0), currents.size(2))\n        spikes = torch.zeros_like(voltage)\n        trace, events = [], []\n        leak = self.leak_logits.sigmoid()\n        threshold = F.softplus(self.threshold_logits) + 1e-4\n        for current, valid in zip(currents.unbind(1), mask.unbind(1)):\n            candidate = leak * voltage + current + self.recurrent(spikes)\n            fired = HardSpike.apply(candidate - threshold)\n            valid = valid.unsqueeze(-1)\n            voltage = torch.where(valid, candidate - threshold * fired.detach(), voltage)\n            spikes = torch.where(valid, fired, spikes)\n            trace.append(torch.where(valid, voltage, 0))\n            events.append(torch.where(valid, spikes, 0))\n        membrane = torch.stack(trace, dim=1)\n        events = torch.stack(events, dim=1)\n        attention = self.attention(membrane).squeeze(-1).masked_fill(~mask, -torch.inf).softmax(1)\n        prediction = self.readout((events * attention.unsqueeze(-1)).sum(1)).squeeze(-1)\n        return prediction, membrane, attention, events\n\n    def forward(self, x, mask):\n        return self.signals(x, mask)[0]\n\n\ndef teacher_for(root, subset, seed, train_hash, validation_hash, cap, device):\n    folder = root / subset / f"seed_{seed}" / "attention_bilstm"\n    metrics = json.loads((folder / "validation_metrics.json").read_text())\n    path = folder / f"epoch_{metrics[\'selected_epoch\']:04d}.pt"\n    checkpoint = torch.load(path, map_location="cpu", weights_only=True)\n    config = checkpoint["config"]\n    sources = config["source_sha256"]\n    if (config != metrics["config"] or checkpoint["epoch"] != metrics["selected_epoch"]\n            or (config["subset"], config["seed"], config["cap"]) != (subset, seed, cap)\n            or (sources["train.npz"], sources["validation.npz"]) != (train_hash, validation_hash)):\n        raise ValueError(f"Teacher checkpoint or prepared data mismatch for {subset} seed {seed}")\n    teacher = Teacher(config["features"]).to(device)\n    teacher.load_state_dict(checkpoint["model"])\n    return teacher.eval().requires_grad_(False), digest(path)\n\n\ndef train(model, adapter, teacher, config, directory, train_data, validation, device):\n    directory.mkdir(parents=True, exist_ok=True)\n    optimizer = torch.optim.Adam([*model.parameters(), *adapter.parameters()], lr=config["lr"])\n    generator = torch.Generator().manual_seed(config["seed"])\n    loader = batches(train_data, config["cap"], config["batch_size"], generator)\n    val_loader = batches(validation, config["cap"], config["batch_size"])\n    history = []\n    prior, _ = load_latest(directory)\n    if prior is not None:\n        if prior["config"] != config:\n            raise ValueError(f"Run configuration changed: {directory}")\n        model.load_state_dict(prior["model"])\n        adapter.load_state_dict(prior["adapter"])\n        optimizer.load_state_dict(prior["optimizer"])\n        restore_rng(prior["rng"], generator)\n        history = prior["history"]\n    supervised, output, hidden, attention, sparse = config["loss_weights"]\n    for epoch in range(len(history) + 1, config["epochs"] + 1):\n        start = time.perf_counter()\n        model.train()\n        loss_sum = event_count = event_slots = 0\n        for x, mask, target in loader:\n            x, mask, target = x.to(device), mask.to(device), target.to(device)\n            optimizer.zero_grad(set_to_none=True)\n            pred, membrane, weights, spikes = model.signals(x, mask)\n            rate = spikes.sum() / (mask.sum() * spikes.size(-1))\n            loss = supervised * F.mse_loss(pred, target) + sparse * rate\n            if output:\n                with torch.no_grad():\n                    reference, states, reference_weights = teacher.signals(x, mask)\n                loss = loss + output * F.mse_loss(pred, reference)\n                if hidden:\n                    loss = loss + hidden * F.mse_loss(adapter(membrane[mask]), states[mask])\n                    loss = loss + attention * F.mse_loss(weights[mask], reference_weights[mask])\n            if not torch.isfinite(loss):\n                raise RuntimeError(f"Nonfinite loss at {directory}, epoch {epoch}")\n            loss.backward()\n            nn.utils.clip_grad_norm_([*model.parameters(), *adapter.parameters()], 1.0, error_if_nonfinite=True)\n            optimizer.step()\n            loss_sum += loss.item() * len(x)\n            event_count += spikes.detach().sum().item()\n            event_slots += mask.sum().item() * spikes.size(-1)\n        val = predict_cycles(model, val_loader, device, config["cap"])\n        history.append(dict(epoch=epoch, loss=loss_sum / len(train_data["y"]),\n                            spike_rate=event_count / event_slots, seconds=time.perf_counter() - start,\n                            validation=regression_metrics(validation["y"], val)))\n        save_checkpoint(dict(epoch=epoch, config=config, model=model.state_dict(), adapter=adapter.state_dict(),\n                             optimizer=optimizer.state_dict(), rng=rng_state(generator), history=history), directory)\n        write_json(directory / "history.json", history)\n        print(f\'{directory}: epoch {epoch}/{config["epochs"]}, RMSE {history[-1]["validation"]["RMSE"]:.3f} cycles\', flush=True)\n\n\ndef run(args):\n    device = torch.device(args.device)\n    results = []\n    for subset in args.subsets:\n        prepared = args.data_root / subset\n        data = {split: read_split(prepared, split) for split in ("train", "validation")}\n        manifest = json.loads((prepared / "split_manifest.json").read_text())\n        cap = float(manifest["rul_cap"])\n        train_hash, validation_hash = (digest(prepared / f"{split}.npz") for split in data)\n        for seed in args.seeds:\n            teacher, teacher_hash = teacher_for(args.baselines, subset, seed, train_hash, validation_hash, cap, device)\n            for name in VARIANTS:\n                directory = args.output / subset / f"seed_{seed}" / name\n                config = dict(model=name, subset=subset, seed=seed, epochs=args.epochs, batch_size=128, lr=0.001,\n                              cap=cap, loss_weights=list(VARIANTS[name]), teacher_sha256=teacher_hash,\n                              train_sha256=train_hash, validation_sha256=validation_hash, code_sha256=digest(__file__))\n                random.seed(seed)\n                np.random.seed(seed)\n                torch.manual_seed(seed)\n                model = Student(data["train"]["X"].shape[-1]).to(device)\n                adapter = nn.Linear(48, teacher.readout.in_features).to(device)\n                if args.phase == "train":\n                    train(model, adapter, teacher, config, directory, data["train"], data["validation"], device)\n                latest, _ = load_latest(directory)\n                if latest is None or latest["config"] != config or latest["epoch"] != args.epochs:\n                    raise RuntimeError(f"Complete the matching training run first: {directory}")\n                best = min(latest["history"], key=lambda row: row["validation"]["RMSE"])["epoch"]\n                state = torch.load(directory / f"epoch_{best:04d}.pt", map_location="cpu", weights_only=True)\n                if state["config"] != config or state["epoch"] != best:\n                    raise RuntimeError(f"Selected checkpoint mismatch: {directory}")\n                model.load_state_dict(state["model"])\n                split = "validation" if args.phase == "train" else "test"\n                if split not in data:\n                    data[split] = read_split(prepared, split)\n                    if len(np.unique(data[split]["engine_id"])) != len(data[split]["engine_id"]):\n                        raise ValueError("Official test requires one prediction per engine")\n                prediction = predict_cycles(model, batches(data[split], cap, 128), device, cap)\n                results.append(save_result(directory, config, split, data[split], prediction,\n                                           best, sum(p.numel() for p in model.parameters())))\n    write_json(args.output / f"{args.phase}_summary.json", aggregate(results))\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    for path in ("data-root", "baselines", "output"):\n        parser.add_argument(f"--{path}", type=Path, required=True)\n    parser.add_argument("--subsets", nargs="+", choices=["FD001", "FD002", "FD003", "FD004"],\n                        default=["FD001", "FD002", "FD003", "FD004"])\n    parser.add_argument("--seeds", nargs="+", type=int, default=[42, 43, 44])\n    parser.add_argument("--epochs", type=int, default=30)\n    parser.add_argument("--phase", choices=["train", "test"], default="train")\n    parser.add_argument("--device", choices=["cpu", "cuda"], default="cuda")\n    args = parser.parse_args()\n    if args.epochs < 1 or args.output.resolve() == args.baselines.resolve():\n        parser.error("Use positive epochs and a separate student output directory")\n    run(args)\n'
(repo / "distill.py").write_text(student_source, encoding="utf-8")


The same student architecture is used in all four variants. The comparison adds output matching, then hidden-state and attention matching, then a firing-rate penalty. The teacher stays frozen. Checkpoint selection uses capped validation RMSE.


In [ ]:
common = ["--data-root", str(data_root), "--subsets", *subsets, "--seeds", *seeds,
          "--epochs", epochs, "--device", "cuda"]
student_command = [sys.executable, "distill.py", *common, "--baselines", str(baselines), "--output", str(output)]
run(*student_command, "--phase", "train")


The next cell checks that every planned validation result exists, then evaluates the frozen models on the official test trajectories. The comparison uses raw official RUL in cycles. Capped metrics remain in the JSON summaries for reference.


In [ ]:
def complete_validation(folder, models_per_subset):
    summary = json.loads((folder / "train_summary.json").read_text())
    for subset in subsets:
        entries = [r for r in summary if r["subset"] == subset and r["target"] == "capped"]
        assert len({r["model"] for r in entries}) == models_per_subset, f"Incomplete validation: {folder}, {subset}"
        assert all(r["seeds"] == list(map(int, seeds)) for r in entries)

complete_validation(baselines, 12)
complete_validation(output, 4)
run(sys.executable, "run_baselines.py", *common, "--output", str(baselines), "--phase", "test")
run(*student_command, "--phase", "test")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def table(folder, filename, target):
    rows = json.loads((folder / filename).read_text())
    return pd.DataFrame([
        {"subset": row["subset"], "model": row["model"], **{
            f"{metric}_{stat}": row[metric][stat] if row[metric] is not None else None
            for metric in ("MAE", "MSE", "RMSE", "R2", "cmapss_score")
            for stat in ("mean", "sample_sd")
        }} for row in rows if row["target"] == target
    ])

validation = pd.concat([table(folder, "train_summary.json", "capped") for folder in (baselines, output)])
test = pd.concat([table(folder, "test_summary.json", "raw") for folder in (baselines, output)])
comparison = validation.merge(test, on=["subset", "model"], suffixes=("_validation_capped", "_test_raw"))
comparison.to_csv(output / "final_comparison.csv", index=False)
display(comparison.sort_values(["subset", "RMSE_mean_test_raw"]))

fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
for subset, ax in zip(subsets, axes.flat):
    subset_rows = test[test["subset"] == subset].sort_values("RMSE_mean")
    ax.barh(subset_rows.model, subset_rows.RMSE_mean, xerr=subset_rows.RMSE_sample_sd.fillna(0))
    ax.invert_yaxis()
    ax.set(title=subset, xlabel="Official test RMSE (cycles), mean ± sample SD")
fig.savefig(output / "test_rmse.png", dpi=160)

errors = []
for subset in subsets:
    for seed in seeds:
        file = output / subset / f"seed_{seed}" / "spikedistill" / "test_predictions.csv"
        predictions = pd.read_csv(file)
        predictions["error"] = predictions.prediction_cycles - predictions.target_raw_cycles
        predictions["subset"], predictions["seed"] = subset, seed
        errors.append(predictions)
residuals = pd.concat(errors, ignore_index=True)
residuals.to_csv(output / "spikedistill_test_residuals.csv", index=False)
fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)
for subset, ax in zip(subsets, axes.flat):
    rows = residuals[residuals.subset == subset]
    ax.scatter(rows.target_raw_cycles, rows.error, s=7, alpha=0.35)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(title=subset, xlabel="True RUL (cycles)", ylabel="Prediction − truth (cycles)")
fig.savefig(output / "test_residuals.png", dpi=160)
